# Early Intervention Triage (`First Milestones`) — Property Graph, ISO GQL & Equity Audit

**Agents for Impact 2026 · Companion to `c_ei_01_load_explore.ipynb` (Section 13+)**

Run this notebook in **Colab Enterprise** or locally via Jupyter / `nbclient` right after `c_ei_01_load_explore.ipynb` (`a4i_early_intervention`) or `bash scripts/load_multistate.sh` (`a4i_early_intervention_multistate`). Every query filters cleanly by `STATE` (`MS`, `AL`, `GA`, `TX`, `CA`, or `NY`) and builds/verifies:

1. **`CREATE OR REPLACE PROPERTY GRAPH {DATASET}.ReferralGraph`** over `CareNode` (`pediatric_and_ei_provider_directory`, excluding `provider_geom`), `Neighborhood` (`v_graph_neighborhoods`), `ReferralHandoff`, `REFERS_TO`, `CAN_REFER_TO`, and `SERVES_AREA`.
2. **Prefix-Safe Variable-Length ISO GQL (`-{2,4}->`) Breach Audit** — exposing why naive `-{2,4}->` queries double-count Type B `Vendor -> Vendor` 4-hop bounce prefixes (`34` rows vs. exact `26` statutory breaches across `44` complete episodes per state).
3. **Visual Graph Path Widget (`TO_JSON(p)`)** — rendering multi-hop `paper_fax` bounce chains inside Colab Enterprise.
4. **Prospective `<=45`-Day Compliant Pathway Router (`GRAPH_TABLE` + `ST_DISTANCE`)** — enforcing Medicaid, Part C vendor status, bilingual capability, uncongested/zero-bounce `CAN_REFER_TO` corridors, the **36-month Part C aging-out cliff**, and zero-vehicle mobility guardrails (with zero `audit_*` or `limited_english_rate` leakage).
5. **Counterfactual Single-Hub Fix Simulator** — quantifying delayed child-days saved by upgrading one `paper_fax` coordinator (`SYN-PROV-*-011`) to electronic intake with pre-checked bilingual/Medicaid slot availability.
6. **Post-Output Equity Audit** — joining `audit_*` demographic shares *only* after routing decisions and reporting statewide HRSA NSCH screening strata without 6x tract fan-out.

In [1]:
from datetime import datetime, timedelta, timezone
import os
import subprocess

import google.auth
import google.auth.credentials
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import bigquery
import pandas as pd


class _LocalGcloudCliCredentials(google.auth.credentials.Credentials):
    """Sanctioned local `gcloud auth print-access-token` fallback when ADC JSON is absent."""

    def refresh(self, request=None):
        token = subprocess.check_output(
            ["gcloud", "auth", "print-access-token"], text=True, timeout=10
        ).strip()
        self.token = token
        self.expiry = datetime.now(timezone.utc).replace(tzinfo=None) + timedelta(minutes=45)


def _resolve_bq_auth():
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    env_proj = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
    try:
        creds, adc_proj = google.auth.default(scopes=scopes)
        return creds, (env_proj or (adc_proj or "").strip()), "application_default_credentials"
    except DefaultCredentialsError:
        creds = _LocalGcloudCliCredentials()
        creds.refresh()
        cli_proj = env_proj or subprocess.check_output(
            ["gcloud", "config", "get-value", "project"], text=True, timeout=8
        ).strip()
        return creds, cli_proj, "gcloud_cli_bridge"


VALID_STATES = ("MS", "AL", "GA", "TX", "CA", "NY")
STATE = os.environ.get("A4I_STATE", "MS").strip().upper()
if STATE not in VALID_STATES:
    raise ValueError(f"Unsupported STATE={STATE!r}; choose from {VALID_STATES}")

DATASET = os.environ.get("A4I_DATASET", "a4i_early_intervention_multistate").strip()
LOCATION = os.environ.get("A4I_LOCATION", "US").strip()
_creds, PROJECT, _auth_mode = _resolve_bq_auth()
bq = bigquery.Client(project=PROJECT, location=LOCATION, credentials=_creds)

# Transparently fall back between multi-state and single-state datasets if needed
try:
    bq.get_dataset(f"{PROJECT}.{DATASET}")
except Exception:
    DATASET = "a4i_early_intervention"


def query_df(sql: str) -> pd.DataFrame:
    """Converts BigQuery results to DataFrame in both Colab Enterprise and minimal local venvs."""
    job = bq.query(sql)
    try:
        return job.to_dataframe()
    except Exception:
        return pd.DataFrame([dict(row.items()) for row in job.result()])


print(f"Project   : {PROJECT} (auth: {_auth_mode})")
print(f"Dataset   : {PROJECT}.{DATASET} ({LOCATION})")
print(f"State     : {STATE}  [100% Synthetic CareNode & ReferralHandoff Network | Real US Census ACS Tracts]")

Project   : qwiklabs-gcp-04-670e1c9d4f6d (auth: gcloud_cli_bridge)
Dataset   : qwiklabs-gcp-04-670e1c9d4f6d.a4i_early_intervention_multistate (US)
State     : MS  [100% Synthetic CareNode & ReferralHandoff Network | Real US Census ACS Tracts]


## 1. Declare the `ReferralGraph` Property Graph (`CareNode`, `Neighborhood`, `ReferralHandoff`, `REFERS_TO`, `CAN_REFER_TO`, `SERVES_AREA`)

Notice the BigQuery Graph DDL rules enforced below:
- Unbackticked `dataset.table` inside `NODE TABLES` and `EDGE TABLES`, with unprefixed node aliases (`CareNode`, `Neighborhood`) in `REFERENCES`.
- `LPAD(..., 11, '0')` preserving leading-zero Census tract GEOIDs (`'06...'` in California, `'01...'` in Alabama).
- `PROPERTIES ALL COLUMNS EXCEPT (provider_geom)` (and views excluding `tract_geom`, `limited_english_rate`, and `audit_*`) so vertex/edge properties remain scalar GQL types (`latitude` and `longitude` stay available).

In [2]:
graph_ddl = f"""
CREATE OR REPLACE VIEW {DATASET}.v_graph_neighborhoods AS
SELECT
  LPAD(census_tract_geoid, 11, '0') AS tract_geoid,
  state_code,
  county_name,
  children_under_5_count,
  poverty_rate,
  no_vehicle_rate,
  assistance_rate,
  median_household_income,
  latitude,
  longitude
FROM {DATASET}.acs_under5_tract_demographics;

CREATE OR REPLACE VIEW {DATASET}.v_graph_refers_to_edges AS
SELECT
  referral_event_id,
  source_provider_id,
  target_provider_id,
  state_code,
  case_chain_id,
  hop_sequence,
  referral_date,
  receiving_agency_id,
  handoff_wait_days,
  transmission_channel,
  handoff_reason,
  concern_domain,
  child_age_months,
  LPAD(child_home_tract_geoid, 11, '0') AS child_home_tract_geoid,
  quarterly_referral_volume,
  dropoff_rate_pct,
  IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1.0, 0.0) AS bounce_flag,
  AVG(IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1.0, 0.0)) OVER (
    PARTITION BY source_provider_id, target_provider_id
  ) AS pair_bounce_rate
FROM {DATASET}.deidentified_part_c_referral_logs;

CREATE OR REPLACE VIEW {DATASET}.v_graph_can_refer_to_edges AS
WITH clean_ehr_base AS (
  SELECT
    source_provider_id,
    target_provider_id,
    handoff_wait_days,
    concern_domain,
    ROUND(
      PERCENTILE_CONT(handoff_wait_days, 0.5) OVER (
        PARTITION BY source_provider_id, target_provider_id
      ),
      0
    ) AS median_ehr_wait_days
  FROM {DATASET}.deidentified_part_c_referral_logs
  WHERE transmission_channel = 'ehr_electronic_portal'
    AND NOT STARTS_WITH(handoff_reason, 'Re-Referral Bounce')
),
clean_ehr_top_domain AS (
  SELECT
    source_provider_id,
    target_provider_id,
    ARRAY_AGG(
      concern_domain ORDER BY domain_cnt DESC, concern_domain ASC LIMIT 1
    )[OFFSET(0)] AS primary_concern_domain
  FROM (
    SELECT
      source_provider_id,
      target_provider_id,
      concern_domain,
      COUNT(*) AS domain_cnt
    FROM clean_ehr_base
    GROUP BY source_provider_id, target_provider_id, concern_domain
  )
  GROUP BY source_provider_id, target_provider_id
),
clean_ehr_pairs AS (
  SELECT
    b.source_provider_id,
    b.target_provider_id,
    COUNT(*) AS clean_ehr_events,
    CAST(ANY_VALUE(b.median_ehr_wait_days) AS INT64) AS median_ehr_wait_days,
    td.primary_concern_domain
  FROM clean_ehr_base AS b
  LEFT JOIN clean_ehr_top_domain AS td
    ON b.source_provider_id = td.source_provider_id
   AND b.target_provider_id = td.target_provider_id
  GROUP BY b.source_provider_id, b.target_provider_id, td.primary_concern_domain
),
pair_bounces AS (
  SELECT
    source_provider_id,
    target_provider_id,
    AVG(IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1.0, 0.0)) AS edge_bounce_rate
  FROM {DATASET}.deidentified_part_c_referral_logs
  GROUP BY source_provider_id, target_provider_id
),
target_outbound_bounces AS (
  SELECT
    source_provider_id,
    AVG(IF(STARTS_WITH(handoff_reason, 'Re-Referral Bounce'), 1.0, 0.0)) AS outbound_bounce_rate
  FROM {DATASET}.deidentified_part_c_referral_logs
  GROUP BY source_provider_id
)
SELECT
  CONCAT(src.provider_id, '_TO_', dst.provider_id) AS can_refer_edge_id,
  src.provider_id AS source_provider_id,
  dst.provider_id AS target_provider_id,
  src.state_code,
  CASE
    WHEN src.node_role = 'Screening_Pediatric_Clinic' AND dst.node_role = 'Part_C_Intake_Coordinator' THEN 'clinic_to_coord'
    WHEN src.node_role = 'Part_C_Intake_Coordinator' AND dst.node_role = 'Multidisciplinary_Eval_Hub' THEN 'coord_to_eval'
    WHEN src.node_role = 'Multidisciplinary_Eval_Hub' AND dst.node_role = 'EI_Therapy_Vendor' THEN 'eval_to_vendor'
    WHEN src.node_role = 'Part_C_Intake_Coordinator' AND dst.node_role = 'EI_Therapy_Vendor' THEN 'coord_to_vendor_fasttrack'
  END AS stage_transition,
  IF(ehr.clean_ehr_events > 0, 'observed_clean_corridor', 'prospective_capability') AS relationship_type,
  CAST(
    COALESCE(
      ehr.median_ehr_wait_days,
      dst.median_internal_wait_days + CASE
        WHEN dst.node_role = 'Part_C_Intake_Coordinator' THEN 0
        WHEN dst.node_role = 'Multidisciplinary_Eval_Hub' THEN 5
        ELSE 6
      END
    ) AS INT64
  ) AS expected_step_days,
  COALESCE(ehr.primary_concern_domain, 'ALL') AS domain_specialization,
  ROUND(SAFE_DIVIDE(dst.active_waitlist_count, dst.monthly_slot_capacity), 2) AS target_congestion_ratio,
  ROUND(COALESCE(out_b.outbound_bounce_rate, 0.0), 4) AS target_outbound_bounce_rate,
  ROUND(COALESCE(pb.edge_bounce_rate, 0.0), 4) AS edge_bounce_rate,
  dst.accepts_medicaid AS target_accepts_medicaid,
  dst.bilingual_services_listed AS target_bilingual
FROM {DATASET}.pediatric_and_ei_provider_directory AS src
JOIN {DATASET}.pediatric_and_ei_provider_directory AS dst
  ON src.state_code = dst.state_code
 AND src.provider_id != dst.provider_id
LEFT JOIN clean_ehr_pairs AS ehr
  ON src.provider_id = ehr.source_provider_id
 AND dst.provider_id = ehr.target_provider_id
LEFT JOIN pair_bounces AS pb
  ON src.provider_id = pb.source_provider_id
 AND dst.provider_id = pb.target_provider_id
LEFT JOIN target_outbound_bounces AS out_b
  ON dst.provider_id = out_b.source_provider_id
WHERE dst.part_c_vendor_enrolled = TRUE
  AND (
    (src.node_role = 'Screening_Pediatric_Clinic' AND dst.node_role = 'Part_C_Intake_Coordinator')
    OR (src.node_role = 'Part_C_Intake_Coordinator' AND dst.node_role = 'Multidisciplinary_Eval_Hub')
    OR (src.node_role = 'Multidisciplinary_Eval_Hub' AND dst.node_role = 'EI_Therapy_Vendor')
    OR (src.node_role = 'Part_C_Intake_Coordinator' AND dst.node_role = 'EI_Therapy_Vendor')
  );

CREATE OR REPLACE VIEW {DATASET}.v_graph_serves_area_edges AS
SELECT
  CONCAT(p.provider_id, '_', LPAD(n.census_tract_geoid, 11, '0')) AS serves_edge_id,
  p.provider_id,
  LPAD(n.census_tract_geoid, 11, '0') AS tract_geoid,
  ROUND(ST_DISTANCE(ST_GEOGPOINT(p.longitude, p.latitude), ST_GEOGPOINT(n.longitude, n.latitude)) / 1000.0, 2) AS travel_distance_km,
  CAST(ROUND((ST_DISTANCE(ST_GEOGPOINT(p.longitude, p.latitude), ST_GEOGPOINT(n.longitude, n.latitude)) / 1000.0) * 1.5, 0) AS INT64) AS estimated_travel_minutes,
  (p.node_role IN ('Multidisciplinary_Eval_Hub', 'EI_Therapy_Vendor') AND p.part_c_vendor_enrolled) AS mobile_visit_capable
FROM {DATASET}.pediatric_and_ei_provider_directory AS p
JOIN {DATASET}.acs_under5_tract_demographics AS n
  ON p.state_code = n.state_code;

CREATE OR REPLACE PROPERTY GRAPH {DATASET}.ReferralGraph
  NODE TABLES (
    {DATASET}.pediatric_and_ei_provider_directory AS CareNode
      KEY (provider_id)
      LABEL CareNode
      PROPERTIES ALL COLUMNS EXCEPT (provider_geom),
    {DATASET}.v_graph_neighborhoods AS Neighborhood
      KEY (tract_geoid)
      LABEL Neighborhood
      PROPERTIES ALL COLUMNS
  )
  EDGE TABLES (
    {DATASET}.deidentified_part_c_referral_logs AS ReferralHandoff
      KEY (referral_event_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL ReferralHandoff
      PROPERTIES ALL COLUMNS,
    {DATASET}.v_graph_refers_to_edges AS REFERS_TO
      KEY (referral_event_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL REFERS_TO
      PROPERTIES ALL COLUMNS,
    {DATASET}.v_graph_can_refer_to_edges AS CAN_REFER_TO
      KEY (can_refer_edge_id)
      SOURCE KEY (source_provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (target_provider_id) REFERENCES CareNode (provider_id)
      LABEL CAN_REFER_TO
      PROPERTIES ALL COLUMNS,
    {DATASET}.v_graph_serves_area_edges AS SERVES_AREA
      KEY (serves_edge_id)
      SOURCE KEY (provider_id) REFERENCES CareNode (provider_id)
      DESTINATION KEY (tract_geoid) REFERENCES Neighborhood (tract_geoid)
      LABEL SERVES_AREA
      PROPERTIES ALL COLUMNS
  );
"""
if DATASET != "a4i_early_intervention":
    bq.query(graph_ddl).result()
    print(f"Declared {PROJECT}.{DATASET}.ReferralGraph successfully.")
else:
    existing_g = list(
        bq.query(
            f"SELECT property_graph_name FROM `{PROJECT}.{DATASET}.INFORMATION_SCHEMA.PROPERTY_GRAPHS` "
            "WHERE property_graph_name = 'ReferralGraph'"
        ).result()
    )
    if not existing_g:
        bq.query(graph_ddl).result()
    print(f"Verified {PROJECT}.{DATASET}.ReferralGraph is active.")

Declared qwiklabs-gcp-04-670e1c9d4f6d.a4i_early_intervention_multistate.ReferralGraph successfully.


## 2. Prefix-Safe Variable-Length `-{2,4}->` ISO GQL Breach Audit

In every state, 8 of the 26 breached referral chains follow a **Type B 4-hop Vendor Bounce** (`Clinic -> Coordinator -> Eval Hub -> Capped EI_Therapy_Vendor (025) -> Fast EI_Therapy_Vendor (026..030)`). Because both Hop 3 and Hop 4 land on `node_role = 'EI_Therapy_Vendor'`, a naive `-{2,4}->` query matches **both** the 3-hop prefix and the 4-hop complete path!

Below we enforce:
1. `clinic.state_code = '{STATE}' AND vendor.state_code = '{STATE}'` (multi-state isolation)
2. `hops[OFFSET(0)].hop_sequence = 1` (starts at true episode origin)
3. `(SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1` (correlates `case_chain_id` across every hop)
4. `QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1` (eliminates 3-hop prefix duplicates on 4-hop `Vendor -> Vendor` chains)

In [3]:
breach_gql = f"""
WITH complete_chains AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
          -[hops:ReferralHandoff]->{{2,4}}
          (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
    WHERE clinic.state_code = '{STATE}'
      AND vendor.state_code = '{STATE}'
      AND hops[OFFSET(0)].hop_sequence = 1
      AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
    LET chain_id = hops[OFFSET(0)].case_chain_id,
        total_hops = ARRAY_LENGTH(hops),
        child_age_months = hops[OFFSET(0)].child_age_months,
        concern_domain = hops[OFFSET(0)].concern_domain,
        home_tract = LPAD(hops[OFFSET(0)].child_home_tract_geoid, 11, '0'),
        intake_coord_id = hops[OFFSET(0)].target_provider_id,
        chain_wait_days = (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h),
        fax_hops = (SELECT COUNTIF(h.transmission_channel = 'paper_fax') FROM UNNEST(hops) AS h),
        bounce_reason = (
          SELECT ANY_VALUE(h.handoff_reason)
          FROM UNNEST(hops) AS h
          WHERE STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        ),
        traced_route = (
          SELECT STRING_AGG(
            CONCAT(h.source_provider_id, ' -[', h.transmission_channel, ' ', CAST(h.handoff_wait_days AS STRING), 'd]-> '),
            '' ORDER BY h.hop_sequence
          )
          FROM UNNEST(hops) AS h
        )
    RETURN
      chain_id,
      child_age_months,
      concern_domain,
      home_tract,
      intake_coord_id,
      vendor.provider_id AS final_vendor_id,
      total_hops,
      fax_hops,
      chain_wait_days,
      bounce_reason,
      traced_route
  )
  QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1
)
SELECT
  COUNT(*) OVER () AS total_breached_chains_in_state,
  chain_id,
  home_tract,
  child_age_months,
  ROUND(child_age_months + chain_wait_days / 30.44, 1) AS est_age_at_ifsp_m,
  concern_domain,
  intake_coord_id,
  total_hops,
  fax_hops,
  chain_wait_days,
  bounce_reason,
  CONCAT(traced_route, final_vendor_id) AS full_path
FROM complete_chains
WHERE chain_wait_days > 45
ORDER BY child_age_months DESC, chain_wait_days DESC
LIMIT 12;
"""
breach_df = query_df(breach_gql)
print(f"State {STATE}: {int(breach_df['total_breached_chains_in_state'].iloc[0])} statutory >45-day breaches (0 prefix duplicates). Top 12 near 36-month cliff:")
display(breach_df.drop(columns=['total_breached_chains_in_state']))

State MS: 26 statutory >45-day breaches (0 prefix duplicates). Top 12 near 36-month cliff:


,chain_id,home_tract,child_age_months,est_age_at_ifsp_m,concern_domain,intake_coord_id,total_hops,fax_hops,chain_wait_days,bounce_reason,full_path
0,CHAIN-MS-023,28049003500,32,34.5,Social-Communication / Autism Screen,SYN-PROV-MS-011,4,3,75,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-007 -[paper_fax 14d]-> SYN-PROV-MS...
1,CHAIN-MS-044,28083950600,29,31.2,Feeding & Sensory,SYN-PROV-MS-012,4,3,68,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-004 -[paper_fax 13d]-> SYN-PROV-MS...
2,CHAIN-MS-020,28049003300,29,31.2,Feeding & Sensory,SYN-PROV-MS-012,4,2,67,Re-Referral Bounce: Therapy Cap / Medicaid Cre...,SYN-PROV-MS-004 -[phone_voicemail 11d]-> SYN-P...
3,CHAIN-MS-043,28083950700,28,30.5,Social-Communication / Autism Screen,SYN-PROV-MS-011,4,3,76,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-003 -[paper_fax 14d]-> SYN-PROV-MS...
4,CHAIN-MS-019,28049010201,28,30.4,Social-Communication / Autism Screen,SYN-PROV-MS-011,4,3,73,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-003 -[paper_fax 11d]-> SYN-PROV-MS...
5,CHAIN-MS-018,28049000302,27,29.4,Motor Milestone Delay,SYN-PROV-MS-010,4,3,74,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-002 -[paper_fax 11d]-> SYN-PROV-MS...
6,CHAIN-MS-040,28083950200,25,27.3,Feeding & Sensory,SYN-PROV-MS-012,4,2,70,Re-Referral Bounce: Therapy Cap / Medicaid Cre...,SYN-PROV-MS-008 -[phone_voicemail 13d]-> SYN-P...
7,CHAIN-MS-015,28049001600,24,26.4,Social-Communication / Autism Screen,SYN-PROV-MS-011,4,2,74,Re-Referral Bounce: Therapy Cap / Medicaid Cre...,SYN-PROV-MS-007 -[phone_voicemail 12d]-> SYN-P...
8,CHAIN-MS-039,28083950800,24,26.4,Social-Communication / Autism Screen,SYN-PROV-MS-011,4,3,72,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-007 -[paper_fax 13d]-> SYN-PROV-MS...
9,CHAIN-MS-038,28027950200,23,25.4,Motor Milestone Delay,SYN-PROV-MS-010,4,3,72,Re-Referral Bounce: Evaluator Waitlist Saturat...,SYN-PROV-MS-006 -[paper_fax 13d]-> SYN-PROV-MS...


## 3. Colab Graph Visualization (`TO_JSON(p)`)

Returning `TO_JSON(p)` over `case_chain_id`-correlated `-{3,4}->` trajectories starting at `hop_sequence = 1` activates BigQuery/Colab's native interactive graph renderer while staying well below the 2 MB payload limit.

In [4]:
viz_gql = f"""
GRAPH {DATASET}.ReferralGraph
MATCH p =
  (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
  -[hops:ReferralHandoff]->{{3,4}}
  (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
WHERE clinic.state_code = '{STATE}'
  AND vendor.state_code = '{STATE}'
  AND hops[OFFSET(0)].hop_sequence = 1
  AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
  AND (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h) > 45
RETURN TO_JSON(p) AS breach_paths
LIMIT 15;
"""
viz_df = query_df(viz_gql)
print(f"Rendered {len(viz_df)} JSON property-graph trajectories for {STATE}:")
display(viz_df.head(5))

Rendered 15 JSON property-graph trajectories for MS:


,breach_paths
0,[{'identifier': 'riEAAAANAAAApgAAAAAAAAAArA8AA...
1,[{'identifier': 'riEAAAANAAAApgAAAAAAAAAArA8AA...
2,[{'identifier': 'riEAAAANAAAApgAAAAAAAAAArA8AA...
3,[{'identifier': 'riEAAAANAAAApgAAAAAAAAAArA8AA...
4,[{'identifier': 'riEAAAANAAAApgAAAAAAAAAArA8AA...


## 4. Prospective `<=45`-Day Compliant Pathway Router (`GRAPH_TABLE` + `ST_DISTANCE`)

Finds Medicaid-enrolled, bilingual `Clinic -> Coordinator -> Eval Hub -> Therapy Vendor` routes clearing the 45-day IDEA Part C ceiling (`34 CFR § 303.310`) for the highest-poverty under-5 tract in `{STATE}`, combining `SERVES_AREA` (`<=80 km` local reach) and `CAN_REFER_TO` / `REFERS_TO` with exact `ST_DISTANCE` from `ST_CENTROID(tract_geom)` to flag zero-vehicle NEMT / mobile evaluation needs (strictly excluding `audit_*` and all-zero `limited_english_rate`).

In [5]:
routing_sql = f"""
WITH target_tract AS (
  SELECT
    LPAD(census_tract_geoid, 11, '0') AS tract_geoid,
    county_name,
    poverty_rate,
    no_vehicle_rate,
    tract_geom
  FROM `{PROJECT}.{DATASET}.acs_under5_tract_demographics`
  WHERE state_code = '{STATE}'
  ORDER BY poverty_rate DESC, children_under_5_count DESC
  LIMIT 1
),
local_serves AS (
  SELECT provider_id, tract_geoid, travel_distance_km
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (p:CareNode {{state_code: '{STATE}'}})-[s:SERVES_AREA]->(n:Neighborhood {{state_code: '{STATE}'}})
    WHERE s.travel_distance_km <= 80.0
    RETURN p.provider_id AS provider_id, n.tract_geoid AS tract_geoid, s.travel_distance_km AS travel_distance_km
  )
),
graph_candidates AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (clinic:CareNode {{state_code: '{STATE}', node_role: 'Screening_Pediatric_Clinic', accepts_medicaid: TRUE}})
          -[h1:REFERS_TO]->
          (coord:CareNode {{state_code: '{STATE}', node_role: 'Part_C_Intake_Coordinator', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
          -[h2:REFERS_TO]->
          (eval:CareNode {{state_code: '{STATE}', node_role: 'Multidisciplinary_Eval_Hub', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
          -[h3:REFERS_TO]->
          (vendor:CareNode {{state_code: '{STATE}', node_role: 'EI_Therapy_Vendor', part_c_vendor_enrolled: TRUE, accepts_medicaid: TRUE, bilingual_services_listed: TRUE}})
    WHERE h1.bounce_flag = 0.0 AND h2.bounce_flag = 0.0 AND h3.bounce_flag = 0.0
      AND eval.active_waitlist_count <= eval.monthly_slot_capacity
      AND vendor.active_waitlist_count <= vendor.monthly_slot_capacity
    LET expected_days = coord.median_internal_wait_days + eval.median_internal_wait_days + vendor.median_internal_wait_days + 11
    FILTER expected_days <= 45
    RETURN
      clinic.provider_id AS clinic_id,
      coord.provider_id AS coord_id,
      eval.provider_id AS eval_id,
      vendor.provider_id AS vendor_id,
      expected_days,
      eval.latitude AS eval_lat,
      eval.longitude AS eval_lon,
      vendor.latitude AS vendor_lat,
      vendor.longitude AS vendor_lon
  )
)
SELECT DISTINCT
  t.tract_geoid,
  t.county_name,
  ROUND(t.poverty_rate * 100, 1) AS poverty_pct,
  ROUND(t.no_vehicle_rate * 100, 1) AS no_vehicle_pct,
  g.clinic_id,
  g.coord_id,
  g.eval_id,
  g.vendor_id,
  g.expected_days,
  COALESCE(
    ls_eval.travel_distance_km,
    ROUND(ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.eval_lon, g.eval_lat)) / 1000.0, 1)
  ) AS eval_km,
  COALESCE(
    ls_ven.travel_distance_km,
    ROUND(ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.vendor_lon, g.vendor_lat)) / 1000.0, 1)
  ) AS vendor_km,
  (t.no_vehicle_rate >= 0.08 AND ST_DISTANCE(ST_CENTROID(t.tract_geom), ST_GEOGPOINT(g.eval_lon, g.eval_lat)) > 25000) AS requires_nemt_or_mobile_eval
FROM graph_candidates AS g
CROSS JOIN target_tract AS t
LEFT JOIN local_serves AS ls_eval
  ON ls_eval.provider_id = g.eval_id AND ls_eval.tract_geoid = t.tract_geoid
LEFT JOIN local_serves AS ls_ven
  ON ls_ven.provider_id = g.vendor_id AND ls_ven.tract_geoid = t.tract_geoid
ORDER BY g.expected_days ASC, eval_km ASC
LIMIT 8;
"""
display(query_df(routing_sql))

,tract_geoid,county_name,poverty_pct,no_vehicle_pct,clinic_id,coord_id,eval_id,vendor_id,expected_days,eval_km,vendor_km,requires_nemt_or_mobile_eval
0,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-003,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-028,29,203.7,205.6,True
1,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-007,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-028,29,203.7,205.6,True
2,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-007,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-029,31,203.7,208.4,True
3,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-003,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-029,31,203.7,208.4,True
4,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-008,SYN-PROV-MS-012,SYN-PROV-MS-020,SYN-PROV-MS-028,32,213.4,205.6,True
5,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-004,SYN-PROV-MS-012,SYN-PROV-MS-020,SYN-PROV-MS-028,32,213.4,205.6,True
6,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-007,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-027,33,203.7,210.0,True
7,28027950700,Coahoma County,57.8,20.3,SYN-PROV-MS-003,SYN-PROV-MS-011,SYN-PROV-MS-019,SYN-PROV-MS-027,33,203.7,210.0,True


## 5. Counterfactual Single-Hub Intervention Simulator

What happens if we intervene at the top hidden `Part_C_Intake_Coordinator` bounce hub (`SYN-PROV-*-011`, whose flat `median_internal_wait_days` is just `5` days!) by:
1. Converting outbound `paper_fax` handoffs (`~21.7d` avg) to `ehr_electronic_portal` (`~13.0d` avg), and
2. Pre-verifying bilingual & Medicaid evaluation slot capacity at intake so toddlers route directly into same-city fast evaluation hubs (`017..020`) and fast vendors (`026..030`) instead of bouncing off saturated hubs (`013..016` / `025`)?

In [6]:
counterfactual_sql = f"""
WITH chains AS (
  SELECT *
  FROM GRAPH_TABLE(
    {DATASET}.ReferralGraph
    MATCH (clinic:CareNode {{node_role: 'Screening_Pediatric_Clinic'}})
          -[hops:ReferralHandoff]->{{2,4}}
          (vendor:CareNode {{node_role: 'EI_Therapy_Vendor'}})
    WHERE clinic.state_code = '{STATE}'
      AND vendor.state_code = '{STATE}'
      AND hops[OFFSET(0)].hop_sequence = 1
      AND (SELECT COUNT(DISTINCT h.case_chain_id) FROM UNNEST(hops) AS h) = 1
    LET chain_id = hops[OFFSET(0)].case_chain_id,
        total_hops = ARRAY_LENGTH(hops),
        coord_id = hops[OFFSET(0)].target_provider_id,
        baseline_wait_d = (SELECT SUM(h.handoff_wait_days) FROM UNNEST(hops) AS h),
        fax_savings_d = (
          SELECT IFNULL(SUM(GREATEST(h.handoff_wait_days - 12, 0)), 0)
          FROM UNNEST(hops) AS h
          WHERE h.transmission_channel = 'paper_fax'
            AND NOT STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        ),
        bounce_hop_d = (
          SELECT IFNULL(SUM(h.handoff_wait_days), 0)
          FROM UNNEST(hops) AS h
          WHERE STARTS_WITH(h.handoff_reason, 'Re-Referral Bounce')
        )
    RETURN chain_id, total_hops, coord_id, baseline_wait_d, fax_savings_d, bounce_hop_d
  )
  QUALIFY ROW_NUMBER() OVER (PARTITION BY chain_id ORDER BY total_hops DESC) = 1
)
SELECT
  c.coord_id,
  p.synthetic_organization_name,
  p.median_internal_wait_days AS flat_internal_wait_d,
  COUNT(*) AS total_routed_cases,
  COUNTIF(c.baseline_wait_d > 45) AS baseline_breaches,
  COUNTIF((c.baseline_wait_d - c.fax_savings_d - c.bounce_hop_d) > 45) AS post_fix_breaches,
  SUM(c.fax_savings_d + c.bounce_hop_d) AS total_child_days_saved
FROM chains AS c
JOIN `{PROJECT}.{DATASET}.pediatric_and_ei_provider_directory` AS p
  ON c.coord_id = p.provider_id
 AND p.state_code = '{STATE}'
GROUP BY 1, 2, 3
ORDER BY total_child_days_saved DESC;
"""
display(query_df(counterfactual_sql))

,coord_id,synthetic_organization_name,flat_internal_wait_d,total_routed_cases,baseline_breaches,post_fix_breaches,total_child_days_saved
0,SYN-PROV-MS-011,Indianola Regional Part C Intake & Service Coo...,5,11,7,0,247
1,SYN-PROV-MS-012,Jackson Regional Part C Intake & Service Coord...,6,11,7,0,232
2,SYN-PROV-MS-009,Cleveland Regional Part C Intake & Service Coo...,8,11,6,0,212
3,SYN-PROV-MS-010,Greenville Regional Part C Intake & Service Co...,9,11,6,0,206


## 6. Post-Output Equity Audit & Statewide NSCH Context

**Strict Governance Rule:** `audit_*` demographic shares are never included in routing queries or priority weights. Below we join `audit_*`, `poverty_rate`, and `no_vehicle_rate` strictly *after* generating pathway recommendations, and inspect statewide HRSA NSCH language strata separately (`6` statewide rows, zero tract fan-out).

In [7]:
equity_sql = f"""
WITH episodes AS (
  SELECT
    case_chain_id,
    ANY_VALUE(LPAD(child_home_tract_geoid, 11, '0')) AS child_home_tract_geoid,
    SUM(handoff_wait_days) AS episode_wait_days
  FROM `{PROJECT}.{DATASET}.deidentified_part_c_referral_logs`
  WHERE state_code = '{STATE}'
  GROUP BY case_chain_id
)
SELECT
  IF(e.episode_wait_days > 45, 'Breached (>45d)', 'Timely (<=45d)') AS statutory_status,
  COUNT(*) AS child_episodes,
  ROUND(AVG(e.episode_wait_days), 1) AS mean_wait_days,
  ROUND(AVG(d.poverty_rate) * 100, 1) AS tract_poverty_pct,
  ROUND(AVG(d.no_vehicle_rate) * 100, 1) AS zero_vehicle_pct,
  ROUND(AVG(d.audit_black_share) * 100, 1) AS audit_black_pct,
  ROUND(AVG(d.audit_hispanic_share) * 100, 1) AS audit_hispanic_pct,
  ROUND(AVG(d.audit_indigenous_share) * 100, 1) AS audit_indigenous_pct,
  ROUND(AVG(d.audit_asian_share) * 100, 1) AS audit_asian_pct,
  ROUND(AVG(d.audit_white_share) * 100, 1) AS audit_white_pct
FROM episodes AS e
JOIN `{PROJECT}.{DATASET}.acs_under5_tract_demographics` AS d
  ON e.child_home_tract_geoid = LPAD(d.census_tract_geoid, 11, '0')
 AND d.state_code = '{STATE}'
GROUP BY 1
ORDER BY 1;
"""
print(f"1) Post-Output Neighborhood Equity Audit for {STATE} (ACS Real Tracts joined to Synthetic Episode Outcomes):")
display(query_df(equity_sql))

nsch_sql = f"""
SELECT
  survey_stratum_id,
  geographic_level,
  household_income_fpl_band,
  primary_home_language,
  ROUND(standardized_screening_rate_pct * 100, 1) AS screening_rate_pct,
  ROUND(followup_requested_pct * 100, 1) AS followup_requested_pct
FROM `{PROJECT}.{DATASET}.nsch_developmental_screening_survey`
WHERE state_code = '{STATE}'
  AND geographic_level = 'Statewide'
ORDER BY screening_rate_pct ASC;
"""
print(f"\n2) Statewide HRSA NSCH Developmental Screening Strata for {STATE} (6 Statewide rows — never raw-joined to tracts!):")
display(query_df(nsch_sql))

1) Post-Output Neighborhood Equity Audit for MS (ACS Real Tracts joined to Synthetic Episode Outcomes):


,statutory_status,child_episodes,mean_wait_days,tract_poverty_pct,zero_vehicle_pct,audit_black_pct,audit_hispanic_pct,audit_indigenous_pct,audit_asian_pct,audit_white_pct
0,Breached (>45d),26,72.8,31.1,11.9,75.5,1.6,0.2,0.3,21.8
1,Timely (<=45d),18,29.4,31.5,10.5,79.2,1.9,0.1,0.2,18.1



2) Statewide HRSA NSCH Developmental Screening Strata for MS (6 Statewide rows — never raw-joined to tracts!):


,survey_stratum_id,geographic_level,household_income_fpl_band,primary_home_language,screening_rate_pct,followup_requested_pct
0,NSCH-MS-01,Statewide,<100% FPL,Non-English Primary Household Language,12.3,16.8
1,NSCH-MS-03,Statewide,100-199% FPL,Non-English Primary Household Language,15.4,14.4
2,NSCH-MS-02,Statewide,<100% FPL,English Primary Household Language,16.0,15.2
3,NSCH-MS-04,Statewide,100-199% FPL,English Primary Household Language,19.3,12.6
4,NSCH-MS-05,Statewide,200-399% FPL,English Primary Household Language,25.6,10.4
5,NSCH-MS-06,Statewide,>=400% FPL,English Primary Household Language,33.0,8.1
